In [4]:
"""
Fine-tune CodeBERT for 4-class bug severity classification in Google Colab.

Classes:
- Critical -> 0
- Major    -> 1
- Minor    -> 2
- Trivial  -> 3

Expected files in Colab runtime:
- /content/train.csv
- /content/test.csv
"""

import os
import shutil
import subprocess
import sys
from dataclasses import dataclass
from typing import Dict

import numpy as np


def install_dependencies() -> None:
    """Install required libraries (safe to run multiple times)."""
    packages = ["transformers", "datasets", "evaluate", "scikit-learn", "accelerate"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)


@dataclass
class Config:
    model_name: str = "microsoft/codebert-base"
    train_path: str = "/content/train.csv"
    test_path: str = "/content/test.csv"
    text_column: str = "description"
    label_column: str = "mapped_severity"
    output_dir: str = "/content/codebert_bug_classifier_outputs"
    final_model_dir: str = "/content/codebert_bug_classifier_final"
    max_length: int = 512
    num_train_epochs: int = 3
    train_batch_size: int = 16
    eval_batch_size: int = 16
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    seed: int = 42


LABEL2ID = {
    "Critical": 0,
    "Major": 1,
    "Minor": 2,
    "Trivial": 3,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def load_datasets(cfg: Config):
    """Load train/test CSV files with Hugging Face datasets."""
    from datasets import load_dataset

    data_files = {"train": cfg.train_path, "test": cfg.test_path}
    return load_dataset("csv", data_files=data_files)


def map_labels(ds, cfg: Config):
    """Convert string severity labels to integer IDs."""
    required = {cfg.text_column, cfg.label_column}
    for split in ds:
        missing = required.difference(ds[split].column_names)
        if missing:
            raise ValueError(f"Missing columns in '{split}': {missing}")

    def _map_label(example):
        label_str = str(example[cfg.label_column]).strip()
        if label_str not in LABEL2ID:
            raise ValueError(
                f"Unexpected label '{label_str}'. Expected one of {list(LABEL2ID)}"
            )
        example["label"] = LABEL2ID[label_str]
        return example

    return ds.map(_map_label)


def tokenize_datasets(ds, cfg: Config, tokenizer):
    """Tokenize description text for CodeBERT."""

    def _tokenize(batch):
        return tokenizer(
            batch[cfg.text_column],
            truncation=True,
            padding=True,
            max_length=cfg.max_length,
        )

    tokenized = ds.map(_tokenize, batched=True)
    keep_cols = ["input_ids", "attention_mask", "label"]
    tokenized = tokenized.remove_columns(
        [c for c in tokenized["train"].column_names if c not in keep_cols]
    )
    tokenized.set_format(type="torch")
    return tokenized


def build_compute_metrics():
    """Return a metrics function for Hugging Face Trainer."""
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support

    def compute_metrics(eval_pred) -> Dict[str, float]:
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        acc = accuracy_score(labels, preds)

        return {
            "accuracy": acc,
            "f1_macro": f1,
            "precision_macro": precision,
            "recall_macro": recall,
        }

    return compute_metrics


def train_and_evaluate(cfg: Config):
    """Train model and evaluate on provided test split."""
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    set_seed(cfg.seed)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=4,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    raw_ds = load_datasets(cfg)
    labeled_ds = map_labels(raw_ds, cfg)
    tokenized_ds = tokenize_datasets(labeled_ds, cfg, tokenizer)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_train_epochs,
        per_device_train_batch_size=cfg.train_batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        fp16=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=cfg.learning_rate,
        weight_decay=cfg.weight_decay,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds["train"],
        eval_dataset=tokenized_ds["test"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=build_compute_metrics(),
    )

    trainer.train()

    metrics = trainer.evaluate(tokenized_ds["test"])
    print("\nFinal test metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value}")

    trainer.save_model(cfg.final_model_dir)
    tokenizer.save_pretrained(cfg.final_model_dir)
    return cfg.final_model_dir


def copy_model_to_drive(local_model_dir: str) -> None:
    """
    Mount Google Drive and copy final model folder to:
    /content/drive/MyDrive/CodeBERT_Bug_Classifier
    """
    from google.colab import drive

    drive.mount("/content/drive")
    drive_target = "/content/drive/MyDrive/CodeBERT_Bug_Classifier"
    os.makedirs(os.path.dirname(drive_target), exist_ok=True)
    shutil.copytree(local_model_dir, drive_target, dirs_exist_ok=True)
    print(f"\nModel copied to: {drive_target}")


def main() -> None:
    cfg = Config()
    install_dependencies()
    saved_model_dir = train_and_evaluate(cfg)
    copy_model_to_drive(saved_model_dir)


if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,1.110164,1.084085,0.489250,0.478456,0.482291,0.489250
2,1.044677,1.037377,0.523750,0.525910,0.535609,0.523750
3,0.819409,1.067628,0.531500,0.532475,0.535497,0.531500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


Final test metrics:
eval_loss: 1.0676279067993164
eval_accuracy: 0.5315
eval_f1_macro: 0.5324753409545113
eval_precision_macro: 0.5354966474898589
eval_recall_macro: 0.5315000000000001
eval_runtime: 30.9403
eval_samples_per_second: 129.281
eval_steps_per_second: 8.08
epoch: 3.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Mounted at /content/drive

Model copied to: /content/drive/MyDrive/CodeBERT_Bug_Classifier
